## 2. RAG 必要性

使用亚马逊美国站服饰箱包知识库，对比模型在无资料和有资料时的回答。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None
print(f"API 已配置, 模型使用{LLM_MODEL}" if client else "未配置 API：保留本地步骤，调用模型的单元会跳过")

API 已配置, 模型使用deepseek-v4-flash-0731


### 2.1. 直接提问：模型缺少内部知识

In [2]:
question = "SKU-YG301 瑜伽裤的面料成分是什么？"

if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print(question)

关于 SKU-YG301 瑜伽裤的具体面料成分，我无法直接提供准确信息，因为该编号可能对应特定品牌或批次，且不同厂家使用的面料会有所差异。  

建议你通过以下方式确认：  
1. **查看商品详情页或标签**：通常面料成分会标注在吊牌、洗水标或产品描述中。  
2. **联系客服**：直接咨询购买渠道的客服，询问该SKU的材质构成。  
3. **常见瑜伽裤面料参考**：多数瑜伽裤采用 **锦纶（尼龙）与氨纶（弹性纤维）** 混纺，常见比例约 **80%锦纶 + 20%氨纶**，以提供弹性、亲肤和吸湿排汗性能；也有部分使用 **涤纶、棉、莫代尔** 等混纺。  

如果有更具体的品牌或产品链接，我可以帮你进一步分析。


### 2.2. 提供资料：模型依据证据回答

先直接读取产品规格；后续 Notebook 再自动检索相关片段。

In [3]:
from pathlib import Path

product_file = Path("../data/产品/瑜伽裤-YG301/产品规格.md")
product_text = product_file.read_text(encoding="utf-8")
prompt = f"""请只根据资料回答问题；资料没有答案时请明确说不知道。

资料：
{product_text}

问题：{question}"""

if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print(prompt[:1200])

SKU-YG301 瑜伽裤的面料成分是：75% Nylon 66（锦纶/超细聚酰胺）+ 25% Lycra Spandex（莱卡四面弹氨纶）。


### 2.3. RAG 三步流程

1. **Retrieve**：从知识库找到相关片段
2. **Augment**：把问题和片段放进提示词
3. **Generate**：让模型根据资料组织答案